In [1]:
from langgraph.graph import StateGraph,START,END
from langgraph.graph.message import add_messages,BaseMessage
from langchain_core.messages import HumanMessage,SystemMessage
from langchain_core.runnables import RunnableConfig
from langgraph.store.postgres import PostgresStore
from langgraph.store.base import BaseStore
from dotenv import load_dotenv
from pydantic import BaseModel,Field
from langchain_groq import ChatGroq
import os
from typing import TypedDict,List,Annotated


In [2]:
load_dotenv(dotenv_path=".env", override=True)
GROQ_API_KEY = os.getenv('GROQ_API_KEY')
model = ChatGroq(
    model="llama-3.3-70b-versatile",  
    api_key=GROQ_API_KEY  
)

In [3]:
class MemoryItems(BaseModel):
    text:str =Field(description="Individual user memory")
    is_new:bool =Field(description="True if new, false if duplicate")

In [6]:
class MemoryDecision(BaseModel):
    should_write:bool
    memories:List[MemoryItems] = Field(default_factory=list)


In [7]:
SYSTEM_PROMPT_TEMPLATE = """You are a helpful assistant with memory capabilities.
If user-specific memory is available, use it to personalize 
your responses based on what you know about the user.

Your goal is to provide relevant, friendly, and tailored 
assistance that reflects the user’s preferences, context, and past interactions.

If the user’s name or relevant personal context is available, always personalize your responses by:
    – Always Address the user by name (e.g., "Sure, Nitish...") when appropriate
    – Referencing known projects, tools, or preferences (e.g., "your MCP server python based project")
    – Adjusting the tone to feel friendly, natural, and directly aimed at the user

Avoid generic phrasing when personalization is possible.

Use personalization especially in:
    – Greetings and transitions
    – Help or guidance tailored to tools and frameworks the user uses
    – Follow-up messages that continue from past context

Always ensure that personalization is based only on known user details and not assumed.

In the end suggest 3 relevant further questions based on the current response and user profile

The user’s memory (which may be empty) is provided as: {user_details_content}
"""

In [8]:
MEMORY_PROMPT = """You are responsible for updating and maintaining accurate user memory.

CURRENT USER DETAILS (existing memories):
{user_details_content}

TASK:
- Review the user's latest message.
- Extract user-specific info worth storing long-term (identity, stable preferences, ongoing projects/goals).
- For each extracted item, set is_new=true ONLY if it adds NEW information compared to CURRENT USER DETAILS.
- If it is basically the same meaning as something already present, set is_new=false.
- Keep each memory as a short atomic sentence.
- No speculation; only facts stated by the user.
- If there is nothing memory-worthy, return should_write=false and an empty list.
"""

In [9]:
memory_extractor=model.with_structured_output(MemoryDecision)


In [10]:
class chatstate(TypedDict):
    messages:Annotated[list[BaseModel],add_messages]

In [ ]:
import uuid
def remember_memory(state: chatstate, config: RunnableConfig, store: BaseStore):
    user_id=config['configurable']['user_id']
    namespace=("users","u1")
    items=store.search(namespace)
    existing = "\n".join(it.value.get("data", "") for it in items) if items else "(empty)"
    last_text=state["messages"][-1].content
    decision:MemoryDecision=memory_extractor.invoke(
        [
            SystemMessage(content=MEMORY_PROMPT.format(user_details_content=existing)),
            {"role": "user", "content": last_text},
        ]
    )
    if decision.should_write:
        for mem in decision.memories:
            if mem.is_new and mem.text.strip():
                store.put(namespace, str(uuid.uuid4()), {"data": mem.text.strip()})
                